# 03 - Methode 2 : variables originales

IFT599-1 / IFT799-51 / IFT799-52 - TP1

Membres : Nom Prenom - CIP

Objectif : selectionner automatiquement les meilleures paires de genes
(diapositive 37) et visualiser la separation des classes.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd

from src.config import CLASSES
from src.loading import charger
from src.selection import selectionner
from src.plotting import nuage

X, y = charger()

## Strategie de recherche

Toutes les paires de genes sont evaluees, soit 280 875 combinaisons. Un seul
parcours suffit a determiner a la fois la meilleure paire globale et la
meilleure paire pour chacune des dix paires de classes.

In [ ]:
paire_globale, meilleures = selectionner(X, y, CLASSES)
print("paire globale :", paire_globale)

## Meilleure paire de variables pour toutes les paires de classes

Mesure d'evaluation : le pire Overlap parmi les 10 paires de classes.

In [ ]:
nuage(X, y, CLASSES, paire_globale,
      "Genes originaux - meilleure paire globale",
      "../figures/original/globale.png")

## Meilleure paire de variables pour chaque paire de classes

Mesure d'evaluation : l'Overlap de la paire de classes consideree.

In [ ]:
lignes = []
for (a, b), (cols, o) in sorted(meilleures.items()):
    lignes.append({"paire": f"{a}-{b}", "variables": cols, "overlap": round(o, 3)})

    nuage(X, y, [a, b], cols,
          f"Genes originaux - {a} vs {b}",
          f"../figures/original/{a}_{b}.png")

t = pd.DataFrame(lignes)
t["bien_separees"] = t["overlap"] < 1
t.to_csv("../results/selection_originales.csv", index=False)
t

### Commentaires

Avec deux genes bien choisis, **cinq paires sur dix** passent sous le seuil,
contre une seule avec les 750 variables : KIRC-PRAD (0.445), COAD-PRAD (0.542),
LUAD-PRAD (0.688), COAD-KIRC (0.794) et BRCA-KIRC (0.892).

On separe donc mieux avec 2 variables qu'avec 750. Sur 750 genes, la plupart ne
distinguent pas les classes et ajoutent du bruit a la distance ; en garder deux
bien choisis supprime ce bruit.

La recherche est exhaustive (280 875 paires, environ 4 minutes). Une version
precedente se limitait aux 30 meilleurs genes du classement 1D et ne trouvait
qu'une paire sous le seuil. La raison est que la separation en 2D ne se deduit
pas des separations sur chaque axe : parmi les 15 genes finalement retenus, 14
ne figuraient pas dans ces 30 premiers.

Les genes retenus different selon les classes comparees, ce qui justifie une
mesure d'evaluation propre a chaque paire. La paire globale (4, 108) n'apparait
dans aucune selection par paire de classes, les deux mesures etant differentes.

BRCA-LUAD (1.894) et BRCA-PRAD (1.206) restent les plus difficiles. BRCA est la
classe la plus nombreuse, donc la plus etendue.